# Assignment 1 — training pipeline

Run with `uv run python main.py train`. Start to finish, with no manual steps, this notebook:

1. Loads the 24 football-data.co.uk season files (LaLiga + Segunda, 2014-15 → 2025-26) from `data/`.
2. Cleans them (column selection, renaming, data-quality fixes).
3. Builds the target (`home_win`) and pre-match features (rolling form, Elo, rest).
4. Splits by season (train ≤ 2022-23, validation 2023-24, test 2024-25).
5. Trains the same logistic regression three ways: scikit-learn, a manual PyTorch loop, and `nn.Module` + `torch.optim`.
6. Evaluates them against two baselines and on the betting decision.
7. Saves everything the dashboard needs to `artifacts/`.

Every choice that was made by looking at the data (thresholds, hyperparameters) is fixed below as a constant,
with the evidence that justified it printed again when the notebook runs.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")            # never open a window: this must run unattended
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()                # main.py runs this from the repo root
DATA_DIR = ROOT / "data"
ART_DIR = ROOT / "artifacts"
ART_DIR.mkdir(exist_ok=True)

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## 1. Load the raw season files

`1tempYYZZ.csv` = Primera (SP1), `2tempYYZZ.csv` = Segunda (SP2). Expected rows: 380 per Primera season
(20 teams × 38 rounds / 2), 462 per Segunda season (22 teams × 42 rounds / 2).

In [ ]:
datasets = {}
for path in sorted(DATA_DIR.glob("*temp*.csv")):
    if "(" in path.name:         # ignore accidental duplicates like '1temp1415 (1).csv'
        continue
    datasets[path.stem] = pd.read_csv(path, encoding="utf-8-sig")   # utf-8-sig strips the BOM ('ï»¿Div')

EXPECTED_ROWS = {"1": 380, "2": 462}
for name, df in datasets.items():
    assert len(df) == EXPECTED_ROWS[name[0]], f"{name} has {len(df)} rows"
print(len(datasets), "files loaded, row counts OK")

## 2. Column selection

Decisions (see REPORT.md for the reasoning):

1. **Drop Asian-handicap markets** — a different bet type from the 1X2 market this project is about
   (including Pinnacle's closing AH odds `PCAHH`/`PCAHA`, which notes.txt does not list explicitly).
2. **Drop any column present in fewer than half of the seasons** (≤ 5 of 12) in either division.
3. **Keep a single bookmaker: Pinnacle** — the only bookmaker with pre-closing *and* closing 1X2 odds in
   12/12 seasons in both divisions, and the sharpest (lowest-margin) market price. Other bookmakers and
   market averages/maxima are dropped.

In [ ]:
ASIAN_HANDICAP = [
    "BbAH", "BbAHh", "AHh", "BbMxAHH", "BbAvAHH", "BbMxAHA", "BbAvAHA",
    "GBAHH", "GBAHA", "GBAH", "LBAHH", "LBAHA", "LBAH", "B365AHH", "B365AHA", "B365AH",
    "PAHH", "PAHA", "MaxAHH", "MaxAHA", "AvgAHH", "AvgAHA", "PCAHH", "PCAHA",
]
for name in datasets:
    datasets[name] = datasets[name].drop(columns=ASIAN_HANDICAP, errors="ignore")

MIN_SEASONS = 6   # at least half of the 12 seasons
def n_seasons(col, div):
    return sum(col in df.columns for n, df in datasets.items() if n.startswith(div))

all_cols = sorted(set(c for df in datasets.values() for c in df.columns))
rare = [c for c in all_cols if n_seasons(c, "1") < MIN_SEASONS or n_seasons(c, "2") < MIN_SEASONS]
for name in datasets:
    datasets[name] = datasets[name].drop(columns=rare, errors="ignore")
print(f"Dropped {len(rare)} columns present in < {MIN_SEASONS} seasons")

NON_PINNACLE = [
    "B365H", "B365D", "B365A", "B365CH", "B365CD", "B365CA",
    "B365>2.5", "B365<2.5", "B365C>2.5", "B365C<2.5", "B365CAHH", "B365CAHA",
    "BWH", "BWD", "BWA", "BWCH", "BWCD", "BWCA",
    "WHH", "WHD", "WHA", "WHCH", "WHCD", "WHCA",
    "IWH", "IWD", "IWA", "VCH", "VCD", "VCA",
    "MaxH", "MaxD", "MaxA", "MaxCH", "MaxCD", "MaxCA",
    "Max>2.5", "Max<2.5", "MaxC>2.5", "MaxC<2.5", "MaxCAHH", "MaxCAHA",
    "AvgH", "AvgD", "AvgA", "AvgCH", "AvgCD", "AvgCA",
    "Avg>2.5", "Avg<2.5", "AvgC>2.5", "AvgC<2.5", "AvgCAHH", "AvgCAHA",
    "AHCh",
]
for name in datasets:
    datasets[name] = datasets[name].drop(columns=NON_PINNACLE, errors="ignore")

remaining = sorted(set(c for df in datasets.values() for c in df.columns))
print(len(remaining), "columns left:", remaining)

## 3. Merge into one table and rename

The season is not stored inside the files, so it is added from the file name before stacking.
Column names follow football-data's `notes.txt`.

In [ ]:
dfs = []
for name, df in datasets.items():
    df = df.copy()
    df["Season"] = "20" + name[5:7] + "-" + name[7:9]      # '1temp1415' -> '2014-15'
    dfs.append(df)
matches = pd.concat(dfs, ignore_index=True)

RENAME = {
    "Div": "League Division", "Date": "Match Date", "Time": "Time of match kick off",
    "HomeTeam": "Home Team", "AwayTeam": "Away Team",
    "FTHG": "Full Time Home Team Goals", "FTAG": "Full Time Away Team Goals",
    "FTR": "Full Time Result (H=Home Win, D=Draw, A=Away Win)",
    "HTHG": "Half Time Home Team Goals", "HTAG": "Half Time Away Team Goals",
    "HTR": "Half Time Result (H=Home Win, D=Draw, A=Away Win)",
    "HS": "Home Team Shots", "AS": "Away Team Shots",
    "HST": "Home Team Shots on Target", "AST": "Away Team Shots on Target",
    "HF": "Home Team Fouls Committed", "AF": "Away Team Fouls Committed",
    "HC": "Home Team Corners", "AC": "Away Team Corners",
    "HY": "Home Team Yellow Cards", "AY": "Away Team Yellow Cards",
    "HR": "Home Team Red Cards", "AR": "Away Team Red Cards",
    "PSH": "Pinnacle home win odds", "PSD": "Pinnacle draw odds", "PSA": "Pinnacle away win odds",
    "PSCH": "Pinnacle closing home win odds", "PSCD": "Pinnacle closing draw odds",
    "PSCA": "Pinnacle closing away win odds",
    "P>2.5": "Pinnacle over 2.5 goals", "P<2.5": "Pinnacle under 2.5 goals",
    "PC>2.5": "Pinnacle closing over 2.5 goals", "PC<2.5": "Pinnacle closing under 2.5 goals",
}
matches = matches.rename(columns=RENAME)

RES = "Full Time Result (H=Home Win, D=Draw, A=Away Win)"
HG, AG = "Full Time Home Team Goals", "Full Time Away Team Goals"
ODDS_PRE = ["Pinnacle home win odds", "Pinnacle draw odds", "Pinnacle away win odds"]

# Over/under 2.5 odds: a different market. Not a target, not a feature (market data), not used for profit.
matches = matches.drop(columns=["Pinnacle over 2.5 goals", "Pinnacle under 2.5 goals",
                                "Pinnacle closing over 2.5 goals", "Pinnacle closing under 2.5 goals"])

matches["League Division"] = matches["League Division"].map({"SP1": "First", "SP2": "Second"})
matches["Match Date"] = pd.to_datetime(matches["Match Date"], dayfirst=True, format="mixed")
matches = matches.sort_values("Match Date", kind="mergesort").reset_index(drop=True)
print(matches.shape)

## 4. Kick-off time: checked, then dropped

It is missing for the five oldest seasons. In the seven seasons where it exists, the home-win rate does not
change across the well-populated time slots (chi-square test below), so it is not worth imputing or scraping.

In [ ]:
t = matches.dropna(subset=["Time of match kick off"]).copy()
t["hour"] = t["Time of match kick off"].str[:2].astype(int)
t["home_win"] = (t[RES] == "H").astype(int)
print(t.groupby("hour")["home_win"].agg(["mean", "count"]).round(3))

big = t[t["hour"].isin([13, 15, 17, 18, 19, 20])]
_, p_value, _, _ = chi2_contingency(pd.crosstab(big["hour"], big["home_win"]))
print(f"chi-square p-value (hour vs home win, slots with 300+ matches): {p_value:.3f}")

matches = matches.drop(columns=["Time of match kick off"])

## 5. Data-quality decisions on rows

1. **Reus walkovers (Segunda 2018-19).** CF Reus was expelled by LaLiga on 18 January 2019; its remaining
   21 fixtures were awarded 1-0 to the opponent without being played. They have no half-time score,
   no stats and no odds. They are not football, so they are removed.
2. **2025-26 removed.** Pinnacle odds stop in mid-January 2026 in the source data, so the business metric
   could not be evaluated on it. 2024-25 (complete odds) is the test season instead.
3. **Segunda 2014-15 to 2016-17 removed.** Match statistics are entirely missing for those three seasons.

In [ ]:
reus_walkovers = ((matches["Season"] == "2018-19") & (matches["League Division"] == "Second")
                  & matches["Half Time Home Team Goals"].isna())
walkover_teams = set(matches.loc[reus_walkovers, "Home Team"]) | set(matches.loc[reus_walkovers, "Away Team"])
assert reus_walkovers.sum() == 21 and "Reus Deportiu" in walkover_teams
matches = matches[~reus_walkovers]

na_odds_2526 = (matches.loc[matches["Season"] == "2025-26"]
                .groupby(matches["Match Date"].dt.to_period("M"))["Pinnacle home win odds"]
                .apply(lambda x: x.isna().mean()).round(2))
print("2025-26 share of matches without Pinnacle odds, by month:")
print(na_odds_2526.to_string())
matches = matches[matches["Season"] != "2025-26"]

early_second = (matches["League Division"] == "Second") & matches["Season"].isin(["2014-15", "2015-16", "2016-17"])
matches = matches[~early_second].reset_index(drop=True)

print(matches.shape)
print(pd.crosstab(matches["Season"], matches["League Division"]))

In [ ]:
# Sanity checks: only scattered missing odds remain, no duplicates, result consistent with goals,
# every team plays a full season except Reus and its 2018-19 opponents.
na = matches.isna().sum()
print("Remaining NaN:\n", na[na > 0].to_string())
assert set(na[na > 0].index) <= set(ODDS_PRE + ["Pinnacle closing home win odds",
                                                  "Pinnacle closing draw odds", "Pinnacle closing away win odds"])

assert matches.duplicated(subset=["Match Date", "Home Team", "Away Team"]).sum() == 0

expected = np.where(matches[HG] > matches[AG], "H", np.where(matches[HG] < matches[AG], "A", "D"))
assert (matches[RES] == expected).all()

games = pd.concat([matches[["Season", "League Division", "Home Team"]].rename(columns={"Home Team": "team"}),
                   matches[["Season", "League Division", "Away Team"]].rename(columns={"Away Team": "team"})])
counts = games.groupby(["Season", "League Division", "team"]).size()
odd_counts = counts[~counts.isin([38, 42])]
assert set(odd_counts.index.get_level_values("Season")) == {"2018-19"}
print(f"{len(odd_counts)} team-seasons with a non-standard number of matches, all in 2018-19 (Reus)")

## 6. Target

`home_win = 1` if the home team wins, `0` for a draw or an away win. Binary because the business decision is
binary ("do I back the home team?") and a draw loses that bet exactly like an away win does.

In [ ]:
TARGET = "home_win"
matches[TARGET] = (matches[RES] == "H").astype(int)
print(f"{len(matches)} matches, home-win rate {matches[TARGET].mean():.3f}")
print(matches.groupby("Season")[TARGET].mean().round(3).to_string())

## 7. Rolling form features

Match statistics are only known **after** a match, so they can only enter the model as averages of a team's
**previous** matches. To compute "a team's last 5 matches" regardless of whether it played at home or away,
each match is duplicated into one row per team (`long` format).

Leakage guard: every rolling mean is `shift(1)` first, so a match never sees its own statistics.
The window runs across seasons (a promoted team carries its last matches from the lower division).

In [ ]:
matches["match_id"] = matches.index

STATS = {   # name -> (home column, away column)
    "goals":   (HG, AG),
    "sot":     ("Home Team Shots on Target", "Away Team Shots on Target"),
    "shots":   ("Home Team Shots", "Away Team Shots"),
    "corners": ("Home Team Corners", "Away Team Corners"),
    "fouls":   ("Home Team Fouls Committed", "Away Team Fouls Committed"),
    "yellow":  ("Home Team Yellow Cards", "Away Team Yellow Cards"),
    "red":     ("Home Team Red Cards", "Away Team Red Cards"),
}

def team_view(is_home):
    """One row per match from one team's point of view ('for' = that team, 'against' = its opponent)."""
    side, other = ("Home Team", "Away Team") if is_home else ("Away Team", "Home Team")
    out = pd.DataFrame({"match_id": matches["match_id"], "date": matches["Match Date"],
                        "team": matches[side], "is_home": int(is_home)})
    for name, (hcol, acol) in STATS.items():
        own, opp = (hcol, acol) if is_home else (acol, hcol)
        out[f"{name}_for"] = matches[own].values
        out[f"{name}_against"] = matches[opp].values
    return out

long = pd.concat([team_view(True), team_view(False)], ignore_index=True)
long = long.sort_values(["team", "date"], kind="mergesort").reset_index(drop=True)
assert len(long) == 2 * len(matches)

long["points"] = 3 * (long["goals_for"] > long["goals_against"]) + 1 * (long["goals_for"] == long["goals_against"])

N = 5
rolling_inputs = ["points"] + [f"{s}_{side}" for s in STATS for side in ("for", "against")]
for col in rolling_inputs:
    long[f"{col}_last{N}"] = (long.groupby("team")[col]
                              .transform(lambda x: x.shift(1).rolling(N, min_periods=1).mean()))

long[["team", "date", "goals_for", "goals_for_last5", "points", "points_last5"]].head(8)

### Cold start

The first matches of each team in the data have little or no history. Matches where either team has fewer
than 5 previous matches (2.9%, none in the test season) are excluded from modelling, so every rolling feature
is a full 5-match window. They stay in the dataset as history for later matches.

In [ ]:
long["n_prev"] = long.groupby("team").cumcount()
prev_home = long[long["is_home"] == 1].set_index("match_id")["n_prev"]
prev_away = long[long["is_home"] == 0].set_index("match_id")["n_prev"]
matches["min_prev"] = matches["match_id"].map(pd.concat([prev_home, prev_away], axis=1).min(axis=1))

MIN_PREV = 5
matches["enough_history"] = matches["min_prev"] >= MIN_PREV
print(f"Matches without {MIN_PREV} previous matches for both teams: {(~matches['enough_history']).sum()}")
print(matches[~matches["enough_history"]].groupby(["Season", "League Division"]).size().to_string())

### Rest days

Days since the team's previous **league** match, capped at 14 (international breaks and the summer count as
"fully rested"). Cup and European fixtures are not in the data, so rest is overestimated for teams playing
midweek elsewhere. A flag for teams returning after > 120 days (no cases between 120 and 365 days) was
considered and rejected: only 14 matches qualify, too few to learn a coefficient.

In [ ]:
long["days_since_last"] = long.groupby("team")["date"].diff().dt.days
print(pd.cut(long["days_since_last"], bins=[0, 14, 30, 60, 120, 365, 2000]).value_counts().sort_index().to_string())
long["rest_days"] = long["days_since_last"].clip(upper=14)

## 8. Elo rating (long-term strength)

One number per team, updated match by match in date order. Standard football values, not tuned:

| Parameter | Value | Meaning |
|---|---|---|
| `K` | 20 | how much a rating moves after each match |
| `HOME_ADV` | 60 | rating points added to the home side when computing the expected result |
| `INIT` | 1500 Primera / 1350 Segunda | rating of a team the first time it appears |
| `REGRESS` | 1/3 | at each new season a team moves a third of the way back to its division's mean |

Leakage guard: the rating is **stored before** the match and **updated after** it.

In [ ]:
K, HOME_ADV, REGRESS = 20, 60, 1 / 3
INIT = {"First": 1500, "Second": 1350}
RESULT_SCORE = {"H": 1.0, "D": 0.5, "A": 0.0}

elo, last_season = {}, {}
home_elo_pre, away_elo_pre = [], []
for season, div, ht, at, r in zip(matches["Season"], matches["League Division"],
                                  matches["Home Team"], matches["Away Team"], matches[RES]):
    for team in (ht, at):
        if team not in elo:
            elo[team] = INIT[div]
        elif last_season[team] != season:
            elo[team] = elo[team] + REGRESS * (INIT[div] - elo[team])
        last_season[team] = season

    rh, ra = elo[ht], elo[at]                       # 1. store BEFORE the match (this is the feature)
    home_elo_pre.append(rh)
    away_elo_pre.append(ra)

    expected_home = 1 / (1 + 10 ** ((ra - (rh + HOME_ADV)) / 400))
    change = K * (RESULT_SCORE[r] - expected_home)  # 2. update AFTER the match
    elo[ht], elo[at] = rh + change, ra - change

matches["home_elo"] = home_elo_pre
matches["away_elo"] = away_elo_pre
print("Final Elo top 10:\n", pd.Series(elo).sort_values(ascending=False).head(10).round(0).to_string())

## 9. Back to one row per match: home, away and difference

What decides a match is how the two teams compare, so each feature also gets a `diff_` version
(home − away). The model uses only the differences: including home, away and diff together would be
perfectly collinear.

In [ ]:
team_feats = [f"{c}_last{N}" for c in rolling_inputs] + ["rest_days"]
h = long[long["is_home"] == 1].set_index("match_id")[team_feats].add_prefix("home_")
a = long[long["is_home"] == 0].set_index("match_id")[team_feats].add_prefix("away_")
matches = matches.join(h, on="match_id").join(a, on="match_id")

# alignment check: every match got its own teams' rows
lh = long[long["is_home"] == 1].set_index("match_id")
la = long[long["is_home"] == 0].set_index("match_id")
m = matches.set_index("match_id")
assert (m["Home Team"] == lh.loc[m.index, "team"]).all() and (m["Away Team"] == la.loc[m.index, "team"]).all()

for f in ["elo"] + team_feats:
    matches[f"diff_{f}"] = matches[f"home_{f}"] - matches[f"away_{f}"]

print(matches.groupby(pd.qcut(matches["diff_elo"], 5), observed=True)[TARGET].agg(["mean", "count"]).round(3))

## 10. Time-based split

A random split would let the model learn from matches played after the ones it is evaluated on.
Splitting by season means the model is always evaluated on the future, as it would be used.

- **train** 2014-15 → 2022-23: fit coefficients
- **validation** 2023-24: choose features, regularization, learning rates and the betting threshold
- **test** 2024-25: used once, at the end

In [ ]:
FEATURE_GROUPS = {
    "strength":   ["diff_elo"],
    "form":       ["diff_goals_for_last5", "diff_goals_against_last5", "diff_points_last5",
                   "diff_sot_for_last5", "diff_sot_against_last5"],
    "context":    ["diff_rest_days"],
    "volume":     ["diff_shots_for_last5", "diff_shots_against_last5",
                   "diff_corners_for_last5", "diff_corners_against_last5"],
    "discipline": ["diff_fouls_for_last5", "diff_fouls_against_last5",
                   "diff_yellow_for_last5", "diff_yellow_against_last5",
                   "diff_red_for_last5", "diff_red_against_last5"],
}
ALL_FEATURES = [f for g in FEATURE_GROUPS.values() for f in g]

TRAIN_SEASONS = ["2014-15", "2015-16", "2016-17", "2017-18", "2018-19",
                 "2019-20", "2020-21", "2021-22", "2022-23"]
VAL_SEASONS, TEST_SEASONS = ["2023-24"], ["2024-25"]

model_df = matches[matches["enough_history"]].copy()
train = model_df[model_df["Season"].isin(TRAIN_SEASONS)]
val = model_df[model_df["Season"].isin(VAL_SEASONS)]
test = model_df[model_df["Season"].isin(TEST_SEASONS)]

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5}: {len(d):>5} matches | home_win {d[TARGET].mean():.3f} | "
          f"{d['Match Date'].min().date()} -> {d['Match Date'].max().date()}")
assert train["Match Date"].max() < val["Match Date"].min() < test["Match Date"].min()
assert sum(d[ALL_FEATURES].isna().sum().sum() for d in (train, val, test)) == 0

y_train, y_val, y_test = train[TARGET].values, val[TARGET].values, test[TARGET].values

## 11. Metrics and baselines

Log loss is the main metric (it is what all three methods minimise). Two baselines bracket the result:
a constant prediction (the train home-win rate, "knowing nothing") and Pinnacle's implied probability with
the bookmaker margin removed ("the market").

In [ ]:
def implied_home_prob(df):
    """Pinnacle pre-closing implied P(home win), margin removed by normalising the three 1/odds."""
    inv = 1 / df[ODDS_PRE]
    return (inv.iloc[:, 0] / inv.sum(axis=1)).values

def metrics(y, p):
    return {"log_loss": log_loss(y, p), "brier": brier_score_loss(y, p),
            "auc": roc_auc_score(y, p) if len(np.unique(p)) > 1 else np.nan,
            "accuracy": accuracy_score(y, (p >= 0.5).astype(int))}

def show(name, m):
    print(f"{name:26} | log loss {m['log_loss']:.4f} | Brier {m['brier']:.4f} | "
          f"AUC {m['auc']:.3f} | acc {m['accuracy']:.3f}")

val_has_odds = val[ODDS_PRE].notna().all(axis=1).values
show("Constant (train rate)", metrics(y_val, np.full(len(y_val), y_train.mean())))
show("Pinnacle (market)", metrics(y_val[val_has_odds], implied_home_prob(val[val_has_odds])))

## 12. Method 1: scikit-learn

### Feature selection by group ablation (validation)

Drop one feature group at a time and compare validation log loss. Decided at group level, not feature by
feature, to avoid overfitting a single validation season.

In [ ]:
def val_logloss(features, C):
    sc = StandardScaler().fit(train[features])
    m = LogisticRegression(C=C, max_iter=1000).fit(sc.transform(train[features]), y_train)
    return log_loss(y_val, m.predict_proba(sc.transform(val[features]))[:, 1])

base = val_logloss(ALL_FEATURES, C=1.0)
print(f"{'All features':24} {base:.4f}")
print(f"{'Elo only':24} {val_logloss(['diff_elo'], C=1.0):.4f}")
for group, cols in FEATURE_GROUPS.items():
    ll = val_logloss([f for f in ALL_FEATURES if f not in cols], C=1.0)
    print(f"without {group:16} {ll:.4f} ({ll - base:+.4f})")

Elo carries most of the signal; recent form and shot/corner volume each add a little; discipline and rest days
add nothing. **Final features: strength + form + volume (10).**

### Regularization strength

`C` is the inverse of the L2 strength. Validation log loss is flat for `C` between 0.1 and 10 (10 features,
~6,000 matches: little room to overfit). Among equal results the most regularized is kept: **C = 0.1**.

In [ ]:
FINAL_GROUPS = ["strength", "form", "volume"]
FEATURES = [f for g in FINAL_GROUPS for f in FEATURE_GROUPS[g]]

for C in [0.001, 0.01, 0.1, 1, 10]:
    print(f"C = {C:<6} | val log loss = {val_logloss(FEATURES, C):.4f}")
BEST_C = 0.1

# one scaler, fitted on train only, shared by all three models
scaler = StandardScaler().fit(train[FEATURES])
X_train_s, X_val_s, X_test_s = (scaler.transform(d[FEATURES]) for d in (train, val, test))

# tol=1e-10: sklearn's default tol (1e-4) stops ~0.001 short of the optimum per coefficient;
# with a strict tolerance it lands on the same solution as gradient descent (checked below)
sk_model = LogisticRegression(C=BEST_C, max_iter=10000, tol=1e-10).fit(X_train_s, y_train)
p_sk_val = sk_model.predict_proba(X_val_s)[:, 1]
show("sklearn", metrics(y_val, p_sk_val))

## 13. Method 2: PyTorch, manual loop

Raw tensors, autograd, and a hand-written gradient step, as in the PyTorch introduction notebook.
To optimise the **same objective** as sklearn, the L2 term is added explicitly to the mean BCE:

`loss = mean BCE + λ · Σw²` with `λ = 1 / (2 · C · n_train)` (bias not penalised, as in sklearn).

In [ ]:
X_tr = torch.tensor(X_train_s, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_va = torch.tensor(X_val_s, dtype=torch.float32)
X_te = torch.tensor(X_test_s, dtype=torch.float32)

n_samples, n_features = X_tr.shape
LAMBDA = 1 / (2 * BEST_C * n_samples)
N_EPOCHS = 2000

def forward_clf(X, w, b):
    return torch.sigmoid(X @ w + b)

def bce_loss(y_pred, y):
    y_pred = y_pred.clamp(1e-7, 1 - 1e-7)          # never take log(0)
    return -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()

def train_manual(lr, n_epochs=N_EPOCHS):
    w = torch.zeros(n_features, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    for epoch in range(n_epochs):
        loss = bce_loss(forward_clf(X_tr, w, b), y_tr) + LAMBDA * (w ** 2).sum()
        loss.backward()
        with torch.no_grad():                       # don't track the update itself
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()                              # gradients accumulate otherwise
        b.grad.zero_()
        history.append(loss.item())
    return w, b, history

manual_runs = {}
for lr in [0.01, 0.1, 0.5, 1.0]:
    w, b, hist = train_manual(lr)
    with torch.no_grad():
        ll = log_loss(y_val, forward_clf(X_va, w, b).numpy())
    manual_runs[lr] = hist
    print(f"lr = {lr:<5} | final train loss {hist[-1]:.4f} | val log loss {ll:.4f}")

lr = 0.01 has not converged after 2,000 epochs; 0.1–1.0 reach the same minimum. **lr = 0.5** is used.

In [ ]:
LR_MANUAL = 0.5
w_man, b_man, hist_man = train_manual(LR_MANUAL)
with torch.no_grad():
    p_man_val = forward_clf(X_va, w_man, b_man).numpy()
show("PyTorch manual", metrics(y_val, p_man_val))

## 14. Method 3: PyTorch, `nn.Module` + `torch.optim`

`optimizer.zero_grad()` replaces the manual `.grad.zero_()`, `optimizer.step()` replaces the `torch.no_grad()`
update. L2 via `weight_decay`, applied to the weights only, with `weight_decay = 2λ` because it adds
`wd · w` to the gradient, which is the gradient of `(wd / 2) · Σw²`.

In [ ]:
def build_model(n_features):
    return nn.Sequential(nn.Linear(n_features, 1), nn.Sigmoid())

def train_standard(lr, n_epochs=N_EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    model = build_model(n_features)
    linear = model[0]
    optimizer = torch.optim.Adam([
        {"params": [linear.weight], "weight_decay": 2 * LAMBDA},
        {"params": [linear.bias], "weight_decay": 0.0},
    ], lr=lr)
    loss_fn = nn.BCELoss()
    history = []
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X_tr).squeeze(-1), y_tr)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history

standard_runs = {}
for lr in [0.001, 0.01, 0.1]:
    model, hist = train_standard(lr)
    with torch.no_grad():
        ll = log_loss(y_val, model(X_va).squeeze(-1).numpy())
    standard_runs[lr] = hist
    print(f"lr = {lr:<6} | final train BCE {hist[-1]:.4f} | val log loss {ll:.4f}")

lr = 0.001 needs ~1,500 epochs; lr = 0.1 oscillates early (Adam's steps are too large at first).
**lr = 0.01** is used.

In [ ]:
LR_STANDARD = 0.01
std_model, hist_std = train_standard(LR_STANDARD)
std_model.eval()
with torch.no_grad():
    p_std_val = std_model(X_va).squeeze(-1).numpy()
show("PyTorch standard", metrics(y_val, p_std_val))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lr, h_ in manual_runs.items():
    axes[0].plot(h_, label=f"lr={lr}")
axes[0].set(title="Manual loop (gradient descent)", xlabel="epoch", ylabel="train loss (BCE + L2)")
for lr, h_ in standard_runs.items():
    axes[1].plot(h_, label=f"lr={lr}")
axes[1].set(title="nn.Module + Adam", xlabel="epoch", ylabel="train BCE")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(ART_DIR / "training_curves.png", dpi=120)
plt.close(fig)

## 15. Do the three methods agree?

Same split, same scaled features, same objective. Coefficients and validation metrics should match.

In [ ]:
coefficients = pd.DataFrame({
    "sklearn": list(sk_model.coef_[0]) + [sk_model.intercept_[0]],
    "torch_manual": list(w_man.detach().numpy()) + [b_man.item()],
    "torch_standard": list(std_model[0].weight.detach().numpy()[0]) + [std_model[0].bias.item()],
}, index=FEATURES + ["intercept"])

sk_default = LogisticRegression(C=BEST_C, max_iter=1000).fit(X_train_s, y_train)   # default tol=1e-4
coefficients["sklearn_default_tol"] = list(sk_default.coef_[0]) + [sk_default.intercept_[0]]

print(coefficients.round(4))
print("max |sklearn - manual|   :", np.abs(coefficients["sklearn"] - coefficients["torch_manual"]).max().round(5))
print("max |sklearn - standard| :", np.abs(coefficients["sklearn"] - coefficients["torch_standard"]).max().round(5))
print("max |default tol - manual|:", np.abs(coefficients["sklearn_default_tol"] - coefficients["torch_manual"]).max().round(5))
coefficients.to_csv(ART_DIR / "coefficients.csv")

## 16. The betting decision (validation)

**Rule:** back the home team when the model's probability exceeds the market's by at least `t`:
`p_model − p_Pinnacle ≥ t`. Stake 1 € at the pre-closing Pinnacle home odds: a win pays `odds − 1`,
anything else loses 1 €. In the confusion matrix, "predicted positive" = "bet placed".

The threshold is not chosen by maximising validation profit (the best-looking values are noise on a few dozen
bets). `t = 0.05` is the first threshold from which ROI stops being clearly negative.

In [ ]:
def betting_results(df, p_model, t):
    has_odds = df[ODDS_PRE].notna().all(axis=1).values
    p_market = np.full(len(df), np.nan)
    p_market[has_odds] = implied_home_prob(df[has_odds])
    bet = has_odds & ((p_model - p_market) >= t)
    won = df[TARGET].values == 1
    odds = df["Pinnacle home win odds"].values
    profit = np.where(bet & won, odds - 1, 0.0) + np.where(bet & ~won, -1.0, 0.0)
    n = int(bet.sum())
    return {"t": t, "n_bets": n, "hit_rate": won[bet].mean() if n else np.nan,
            "profit": profit.sum(), "roi": profit.sum() / n if n else np.nan}

thresholds = np.round(np.arange(-0.05, 0.16, 0.01), 2)
val_bets = pd.DataFrame([betting_results(val, p_sk_val, t) for t in thresholds])
print(val_bets.round(3).to_string(index=False))

# where the model disagrees with the market, who is right?
p_mkt = implied_home_prob(val[val_has_odds])
p_mod, y_h = p_sk_val[val_has_odds], y_val[val_has_odds]
for t in [0.0, 0.05, 0.10]:
    sel = (p_mod - p_mkt) >= t
    print(f"t={t:.2f} | n={sel.sum():>3} | model {p_mod[sel].mean():.3f} | market {p_mkt[sel].mean():.3f} | "
          f"realised {y_h[sel].mean():.3f}")

T_FINAL = 0.05

## 17. Final evaluation on test (2024-25), once

In [ ]:
with torch.no_grad():
    preds_test = {
        "sklearn": sk_model.predict_proba(X_test_s)[:, 1],
        "torch_manual": forward_clf(X_te, w_man, b_man).numpy(),
        "torch_standard": std_model(X_te).squeeze(-1).numpy(),
    }
preds_val = {"sklearn": p_sk_val, "torch_manual": p_man_val, "torch_standard": p_std_val}

rows = []
for split, df, y, preds in [("val", val, y_val, preds_val), ("test", test, y_test, preds_test)]:
    has = df[ODDS_PRE].notna().all(axis=1).values
    rows.append({"split": split, "model": "constant", **metrics(y, np.full(len(y), y_train.mean()))})
    rows.append({"split": split, "model": "pinnacle", **metrics(y[has], implied_home_prob(df[has]))})
    for name, p in preds.items():
        bets = betting_results(df, p, T_FINAL)
        rows.append({"split": split, "model": name, **metrics(y, p),
                     **{k: bets[k] for k in ("n_bets", "hit_rate", "profit", "roi")}})
results = pd.DataFrame(rows)
print(results.round(4).to_string(index=False))
results.to_csv(ART_DIR / "results.csv", index=False)

bet = (preds_test["sklearn"] - implied_home_prob(test)) >= T_FINAL
cm = confusion_matrix(y_test, bet.astype(int), labels=[1, 0])
print(pd.DataFrame(cm, index=["home win", "no home win"], columns=["bet", "no bet"]))

## 18. Save artifacts for the dashboard

The app only reads these files; it never trains anything.

In [ ]:
joblib.dump(scaler, ART_DIR / "scaler.joblib")
joblib.dump(sk_model, ART_DIR / "model_sklearn.joblib")
torch.save({"w": w_man.detach(), "b": b_man.detach()}, ART_DIR / "model_torch_manual.pt")
torch.save(std_model.state_dict(), ART_DIR / "model_torch_standard.pt")

keep = ["match_id", "Match Date", "Season", "League Division", "Home Team", "Away Team", RES, TARGET,
        "Pinnacle home win odds", "Pinnacle draw odds", "Pinnacle away win odds", "enough_history"] + ALL_FEATURES
model_df[keep].assign(split=np.select(
    [model_df["Season"].isin(TRAIN_SEASONS), model_df["Season"].isin(VAL_SEASONS)], ["train", "val"], "test")
).to_csv(ART_DIR / "features.csv", index=False)

config = {"features": FEATURES, "feature_groups": FEATURE_GROUPS, "final_groups": FINAL_GROUPS,
          "best_C": BEST_C, "lambda": LAMBDA, "lr_manual": LR_MANUAL, "lr_standard": LR_STANDARD,
          "n_epochs": N_EPOCHS, "t_final": T_FINAL, "train_rate": float(y_train.mean()),
          "train_seasons": TRAIN_SEASONS, "val_seasons": VAL_SEASONS, "test_seasons": TEST_SEASONS}
(ART_DIR / "config.json").write_text(json.dumps(config, indent=2))

print("Saved:", sorted(p.name for p in ART_DIR.iterdir()))